# Three-second magnitude: validation prior experiments

Uses only the corrected-bin checkpoint and the existing validation split. Run from the project root or a subdirectory, with the waveform data mounted at the paths expected by `instancepipelineprior`. Dependencies: PyTorch, NumPy, pandas, SciPy and IPython, plus your two project modules. No notebook cells were executed during this revision; use **Restart Kernel and Run All** in your model environment.

Rates, accuracy, precision, recall and F1 are **fractions from 0 to 1**. Bias is prediction minus truth; underestimation by 0.5 means error **≤ −0.5**. Undefined statistics use nullable `pd.NA`, never a fabricated zero. Balanced-bin MAE averages nonempty groups in `[0,6)`; coverage and counts are explicit. All samples remain in overall and tail metrics, including magnitudes outside those six groups. Research rationale and limitations follow the two result tables.

In [2]:
from pathlib import Path
import torch

from instancepipelineprior import get_data_loaders
from discretizedtrain import (
    EarthquakeCNN,
    magnitude_to_bin,
    MIN_MAGNITUDE,
    MAX_MAGNITUDE,
    BIN_WIDTH,
    NUM_BINS,
)

ROOT = Path.cwd()

BIN_CENTERS = (
    MIN_MAGNITUDE
    + (torch.arange(NUM_BINS, dtype=torch.float32) + 0.5) * BIN_WIDTH
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

checkpoint_path = ROOT / "data" / "best_model_huberandcross5s075fixed.pth"

checkpoint = torch.load(
    checkpoint_path,
    map_location=device,
    weights_only=False,
)

model = EarthquakeCNN().to(device)
model.load_state_dict(checkpoint["model_state_dict"])
model.eval()

print("Device:", device)
print("Loaded:", checkpoint_path)

Train samples: 979487
Validation samples: 96993
Test samples: 82769
Train batches: 7653
Val batches: 758
Test batches: 647
Device: cuda
Loaded: /home/ec2-user/Earthquake/bayesianprior/data/best_model_huberandcross5s075fixed.pth


In [3]:
from torch.utils.data import DataLoader

train_loader, val_loader, test_loader = get_data_loaders()

# Deterministic loader over the full training set for estimating the prior
prior_loader = DataLoader(
    train_loader.dataset,
    batch_size=train_loader.batch_size,
    shuffle=False,
    drop_last=False,
    num_workers=0,
    collate_fn=train_loader.collate_fn,
)

In [4]:
from scipy.ndimage import gaussian_filter1d

In [5]:
def compute_empirical_prior(dataloader):
    counts = torch.zeros(NUM_BINS, dtype=torch.long)

    for _, magnitudes in dataloader:
        magnitudes = magnitudes.detach().cpu().reshape(-1)
        indices = magnitude_to_bin(magnitudes)
        counts += torch.bincount(indices, minlength=NUM_BINS)

    prior = counts.to(torch.float64) / counts.sum()
    return counts, prior


train_counts, empirical_prior = compute_empirical_prior(prior_loader)

sigma = 1.0
smoothed_prior = torch.from_numpy(gaussian_filter1d(empirical_prior.numpy(), sigma=sigma, mode="reflect", truncate=4.0))
smoothed_prior = smoothed_prior / smoothed_prior.sum()

In [6]:
BIN_CENTERS = MIN_MAGNITUDE + (torch.arange(NUM_BINS, dtype=torch.float64) + 0.5) * BIN_WIDTH

In [13]:
import math

import torch
import torch.nn.functional as F

from scipy.stats import spearmanr


def collect_logits(model, dataloader, device):
    all_logits, all_targets = [], []
    model.eval()

    with torch.inference_mode():
        for waveforms, magnitudes in dataloader:
            logits = model(waveforms.to(device))
            all_logits.append(logits.cpu().to(torch.float64))
            all_targets.append(magnitudes.cpu().reshape(-1))

    return torch.cat(all_logits), torch.cat(all_targets)


val_logits, val_targets = collect_logits(model, val_loader, device)

val_class_indices = magnitude_to_bin(val_targets)
val_targets = val_targets.to(torch.float64)

raw_probs = F.softmax(val_logits, dim=1)
raw_predictions = raw_probs @ BIN_CENTERS

normalized_entropy = -(raw_probs * torch.log(raw_probs + 1e-12)).sum(dim=1) / math.log(NUM_BINS)

magnitude_threshold = 4.0
tail_probability = raw_probs[:, BIN_CENTERS >= magnitude_threshold].sum(dim=1)

absolute_raw_error = (raw_predictions - val_targets).abs()

correlation = spearmanr(normalized_entropy.numpy(), absolute_raw_error.numpy())
entropy_error_spearman = float(correlation.statistic)
entropy_error_pvalue = float(correlation.pvalue)

In [14]:
import numpy as np
import torch


def calculate_metrics(predictions, targets):
    if targets.numel() == 0:
        return dict.fromkeys(("mae", "rmse", "bias", "accuracy_0.2", "underestimate_rate", "underestimate_0.5_rate"))

    errors = predictions - targets

    return {
        "mae": errors.abs().mean().item(),
        "rmse": errors.square().mean().sqrt().item(),
        "bias": errors.mean().item(),
        "accuracy_0.2": (errors.abs() <= 0.2).double().mean().item(),
        "underestimate_rate": (errors < 0).double().mean().item(),
        "underestimate_0.5_rate": (errors <= -0.5).double().mean().item()
    }


def extreme_metrics(probs, targets):
    predictions = probs @ BIN_CENTERS
    errors = predictions - targets
    raw_errors = raw_predictions - targets
    metrics = {}

    for threshold, prefix in ((4.0, "extreme40"), (4.5, "extreme45"), (5.0, "extreme50")):
        mask = targets >= threshold
        group = calculate_metrics(predictions[mask], targets[mask])

        metrics[f"{prefix}_count"] = int(mask.sum())
        metrics.update({f"{prefix}_{key}": value for key, value in group.items()})

        severe_before = mask & (raw_errors <= -0.5)
        newly_severe = mask & (raw_errors > -0.5)

        metrics[f"{prefix}_raw_severe_count"] = int(severe_before.sum())
        metrics[f"{prefix}_recovered_count"] = int((severe_before & (errors > -0.5)).sum())
        metrics[f"{prefix}_recovery_rate"] = (errors[severe_before] > -0.5).double().mean().item() if severe_before.any() else None
        metrics[f"{prefix}_new_severe_count"] = int((newly_severe & (errors <= -0.5)).sum())
        metrics[f"{prefix}_mean_abs_error_change"] = (errors[mask].abs() - raw_errors[mask].abs()).mean().item() if mask.any() else None
        metrics[f"{prefix}_p95_abs_error"] = torch.quantile(errors[mask].abs(), 0.95).item() if mask.any() else None

    for threshold, prefix in ((4.0, "m4"), (5.0, "m5")):
        probability = probs[:, BIN_CENTERS >= threshold].sum(dim=1)
        truth = targets >= threshold

        metrics[f"{prefix}_probability_brier"] = (probability - truth.double()).square().mean().item()
        metrics[f"{prefix}_mean_probability"] = probability.mean().item()
        metrics[f"{prefix}_observed_frequency"] = truth.double().mean().item()

        for cutoff in (0.1, 0.5):
            positive = probability >= cutoff
            tp = int((positive & truth).sum())
            fp = int((positive & ~truth).sum())
            fn = int((~positive & truth).sum())
            label = f"{prefix}_prob{int(100 * cutoff)}"

            metrics.update({
                f"{label}_TP": tp,
                f"{label}_FP": fp,
                f"{label}_FN": fn,
                f"{label}_precision": tp / (tp + fp) if tp + fp else None,
                f"{label}_recall": tp / (tp + fn) if tp + fn else None
            })

    return metrics


def evaluate_probabilities(probs, targets):
    predictions = probs @ BIN_CENTERS
    overall = calculate_metrics(predictions, targets)

    row = {
        f"overall_{key}" if key in ("mae", "rmse", "bias") else key: value
        for key, value in overall.items()
    }

    row["overall_count"] = targets.numel()

    bin_maes = []

    for lower in range(6):
        mask = (targets >= lower) & (targets < lower + 1)
        metrics = calculate_metrics(predictions[mask], targets[mask])

        row[f"count_{lower}_{lower + 1}"] = int(mask.sum())
        row[f"mae_{lower}_{lower + 1}"] = metrics["mae"]
        row[f"bias_{lower}_{lower + 1}"] = metrics["bias"]

        if metrics["mae"] is not None:
            bin_maes.append(metrics["mae"])

    row["balanced_bin_mae"] = float(np.mean(bin_maes)) if bin_maes else None
    row["balanced_bin_groups_present"] = len(bin_maes)
    row["outside_broad_bins_count"] = int(((targets < 0) | (targets >= 6)).sum())

    for threshold, prefix in ((3.5, "tail35"), (4.0, "tail40")):
        mask = targets >= threshold
        metrics = calculate_metrics(predictions[mask], targets[mask])

        row[f"{prefix}_count"] = int(mask.sum())
        row.update({f"{prefix}_{key}": value for key, value in metrics.items() if key != "accuracy_0.2"})

    predicted_positive = predictions >= 4
    true_positive = targets >= 4

    tp = int((predicted_positive & true_positive).sum())
    fp = int((predicted_positive & ~true_positive).sum())
    fn = int((~predicted_positive & true_positive).sum())

    row.update({
        "m4_TP": tp,
        "m4_FP": fp,
        "m4_FN": fn,
        "m4_precision": tp / (tp + fp) if tp + fp else None,
        "m4_recall": tp / (tp + fn) if tp + fn else None,
        "m4_f1": 2 * tp / (2 * tp + fp + fn) if 2 * tp + fp + fn else None
    })

    row.update(extreme_metrics(probs, targets))
    return row


raw_metrics = evaluate_probabilities(raw_probs, val_targets)

## A. Entropy-gated smoothed prior

In [15]:
import math
import pandas as pd
import torch
import torch.nn.functional as F


def apply_entropy_prior(logits, prior, bin_centers, alpha_max=0.4, power=1.0, magnitude_threshold=4.0, gate="entropy_tail"):
    log_model = F.log_softmax(logits, dim=1)
    probs = log_model.exp()

    entropy = -(probs * log_model).sum(dim=1) / math.log(logits.shape[1])
    tail = probs[:, bin_centers >= magnitude_threshold].sum(dim=1)

    alpha = torch.full_like(entropy, alpha_max)

    if gate in ("entropy_only", "entropy_tail"):
        alpha *= entropy.pow(power)

    if gate in ("tail_only", "entropy_tail"):
        alpha *= (1 - tail)

    posterior = F.softmax((1 - alpha[:, None]) * log_model + alpha[:, None] * prior.log(), dim=1)

    return posterior, entropy, alpha, tail


alpha_max_values = [0.0, 0.1, 0.2, 0.3, 0.4]
power_values = [0.5, 1.0, 2.0]
entropy_rows = []

for alpha_max in alpha_max_values:
    for power in ([1.0] if alpha_max == 0 else power_values):
        probs, entropy, alpha, tail = apply_entropy_prior(
            val_logits, smoothed_prior, BIN_CENTERS, alpha_max, power, magnitude_threshold
        )

        metrics = raw_metrics.copy() if alpha_max == 0 else evaluate_probabilities(probs, val_targets)

        row = {
            "method": "raw" if alpha_max == 0 else "entropy_prior",
            "gate": "entropy_tail",
            "alpha_max": alpha_max,
            "power": power,
            "magnitude_threshold": magnitude_threshold,
            **metrics,
            "entropy_error_spearman": entropy_error_spearman,
            "entropy_error_pvalue": entropy_error_pvalue,
            "mean_alpha": alpha.mean().item(),
            "mean_tail_probability": tail.mean().item()
        }

        for lower in range(6):
            mask = (val_targets >= lower) & (val_targets < lower + 1)
            row[f"mean_entropy_{lower}_{lower + 1}"] = entropy[mask].mean().item() if mask.any() else None

        entropy_rows.append(row)


for alpha_max in (0.2, 0.4):
    for gate in ("constant", "entropy_only", "tail_only"):
        probs, entropy, alpha, tail = apply_entropy_prior(
            val_logits, smoothed_prior, BIN_CENTERS, alpha_max, 1.0, magnitude_threshold, gate
        )

        row = {
            "method": "gate_ablation",
            "gate": gate,
            "alpha_max": alpha_max,
            "power": 1.0,
            "magnitude_threshold": magnitude_threshold,
            **evaluate_probabilities(probs, val_targets),
            "entropy_error_spearman": entropy_error_spearman,
            "entropy_error_pvalue": entropy_error_pvalue,
            "mean_alpha": alpha.mean().item(),
            "mean_tail_probability": tail.mean().item()
        }

        for lower in range(6):
            mask = (val_targets >= lower) & (val_targets < lower + 1)
            row[f"mean_entropy_{lower}_{lower + 1}"] = entropy[mask].mean().item() if mask.any() else None

        entropy_rows.append(row)


entropy_results_df = pd.DataFrame(entropy_rows)

## B. Supported-class post-hoc logit adjustment

In [17]:
import math
import pandas as pd
import torch
import torch.nn.functional as F

from scipy.ndimage import gaussian_filter1d


def apply_logit_adjustment(logits, prior, tau):
    supported = prior > 0
    adjusted_logits = torch.full_like(logits, -torch.inf)
    adjusted_logits[:, supported] = logits[:, supported] - tau * prior[supported].log()
    return F.softmax(adjusted_logits, dim=1)


def smoothed_supported_prior(sigma):
    supported = empirical_prior > 0
    prior = torch.from_numpy(gaussian_filter1d(empirical_prior.numpy(), sigma=sigma, mode="reflect", truncate=4.0))
    prior[~supported] = 0
    return prior / prior.sum()


def capped_logit_adjustment(logits, prior, tau, boost_cap_ratio):
    supported = prior > 0
    boost = tau * (prior[supported].max().log() - prior[supported].log())
    boost = boost.clamp(max=math.log(boost_cap_ratio))

    adjusted = torch.full_like(logits, -torch.inf)
    adjusted[:, supported] = logits[:, supported] + boost

    return F.softmax(adjusted, dim=1)


def group_logit_adjustment(logits, prior, tau, threshold):
    supported = prior > 0
    tail = BIN_CENTERS >= threshold
    tail_prior = prior[tail].sum()
    group_prior = torch.where(tail, tail_prior, 1 - tail_prior)

    adjusted = torch.full_like(logits, -torch.inf)
    adjusted[:, supported] = logits[:, supported] - tau * group_prior[supported].log()

    return F.softmax(adjusted, dim=1)


def argmax_metrics(probs, targets):
    predictions = BIN_CENTERS[probs.argmax(dim=1)]

    overall = calculate_metrics(predictions, targets)
    tail = calculate_metrics(predictions[targets >= 4], targets[targets >= 4])

    return {
        "argmax_overall_mae": overall["mae"],
        "argmax_M4_mae": tail["mae"],
        "argmax_M4_bias": tail["bias"]
    }


tau_values = [0.0, 0.05, 0.1, 0.15, 0.2, 0.25, 0.3, 0.4, 0.5, 0.75, 1.0]

logit_rows = [{
    "method": "raw",
    "tau": 0.0,
    **raw_metrics,
    **argmax_metrics(raw_probs, val_targets)
}]


for tau in tau_values:
    probs = apply_logit_adjustment(val_logits, empirical_prior, tau)

    logit_rows.append({
        "method": "supported_tau_0" if tau == 0 else "logit_adjustment",
        "tau": tau,
        **evaluate_probabilities(probs, val_targets),
        **argmax_metrics(probs, val_targets)
    })


for prior_sigma in (0.5, 1.0, 2.0):
    prior = smoothed_supported_prior(prior_sigma)

    for tau in (0.1, 0.25, 0.5):
        probs = apply_logit_adjustment(val_logits, prior, tau)

        logit_rows.append({
            "method": "smoothed_inverse",
            "tau": tau,
            "prior_sigma": prior_sigma,
            **evaluate_probabilities(probs, val_targets),
            **argmax_metrics(probs, val_targets)
        })


for boost_cap_ratio in (2.0, 5.0):
    for tau in (0.25, 0.5, 1.0):
        probs = capped_logit_adjustment(val_logits, empirical_prior, tau, boost_cap_ratio)

        logit_rows.append({
            "method": "capped_inverse",
            "tau": tau,
            "boost_cap_ratio": boost_cap_ratio,
            **evaluate_probabilities(probs, val_targets),
            **argmax_metrics(probs, val_targets)
        })


for group_threshold in (3.5, 4.0):
    for tau in (0.1, 0.25, 0.5, 1.0):
        probs = group_logit_adjustment(val_logits, empirical_prior, tau, group_threshold)

        logit_rows.append({
            "method": "group_inverse",
            "tau": tau,
            "group_threshold": group_threshold,
            **evaluate_probabilities(probs, val_targets),
            **argmax_metrics(probs, val_targets)
        })


logit_adjustment_results_df = pd.DataFrame(logit_rows)

In [18]:
import pandas as pd

with pd.option_context(
    "display.max_rows", None,
    "display.max_columns", None,
    "display.float_format", "{:.6f}".format
):
    display(entropy_results_df)
    display(logit_adjustment_results_df)

,method,gate,alpha_max,power,magnitude_threshold,overall_mae,overall_rmse,overall_bias,accuracy_0.2,underestimate_rate,underestimate_0.5_rate,overall_count,count_0_1,mae_0_1,bias_0_1,count_1_2,mae_1_2,bias_1_2,count_2_3,mae_2_3,bias_2_3,count_3_4,mae_3_4,bias_3_4,count_4_5,mae_4_5,bias_4_5,count_5_6,mae_5_6,bias_5_6,balanced_bin_mae,balanced_bin_groups_present,outside_broad_bins_count,tail35_count,tail35_mae,tail35_rmse,tail35_bias,tail35_underestimate_rate,tail35_underestimate_0.5_rate,tail40_count,tail40_mae,tail40_rmse,tail40_bias,tail40_underestimate_rate,tail40_underestimate_0.5_rate,m4_TP,m4_FP,m4_FN,m4_precision,m4_recall,m4_f1,extreme40_count,extreme40_mae,extreme40_rmse,extreme40_bias,extreme40_accuracy_0.2,extreme40_underestimate_rate,extreme40_underestimate_0.5_rate,extreme40_raw_severe_count,extreme40_recovered_count,extreme40_recovery_rate,extreme40_new_severe_count,extreme40_mean_abs_error_change,extreme40_p95_abs_error,extreme45_count,extreme45_mae,extreme45_rmse,extreme45_bias,extreme45_accuracy_0.2,extreme45_underestimate_rate,extreme45_underestimate_0.5_rate,extreme45_raw_severe_count,extreme45_recovered_count,extreme45_recovery_rate,extreme45_new_severe_count,extreme45_mean_abs_error_change,extreme45_p95_abs_error,extreme50_count,extreme50_mae,extreme50_rmse,extreme50_bias,extreme50_accuracy_0.2,extreme50_underestimate_rate,extreme50_underestimate_0.5_rate,extreme50_raw_severe_count,extreme50_recovered_count,extreme50_recovery_rate,extreme50_new_severe_count,extreme50_mean_abs_error_change,extreme50_p95_abs_error,m4_probability_brier,m4_mean_probability,m4_observed_frequency,m4_prob10_TP,m4_prob10_FP,m4_prob10_FN,m4_prob10_precision,m4_prob10_recall,m4_prob50_TP,m4_prob50_FP,m4_prob50_FN,m4_prob50_precision,m4_prob50_recall,m5_probability_brier,m5_mean_probability,m5_observed_frequency,m5_prob10_TP,m5_prob10_FP,m5_prob10_FN,m5_prob10_precision,m5_prob10_recall,m5_prob50_TP,m5_prob50_FP,m5_prob50_FN,m5_prob50_precision,m5_prob50_recall,entropy_error_spearman,entropy_error_pvalue,mean_alpha,mean_tail_probability,mean_entropy_0_1,mean_entropy_1_2,mean_entropy_2_3,mean_entropy_3_4,mean_entropy_4_5,mean_entropy_5_6
0,raw,entropy_tail,0.000000,1.000000,4.000000,0.363262,0.488058,0.085835,0.384409,0.411566,0.091543,96993,5190,0.738733,0.738733,20718,0.425762,0.381808,60450,0.274630,0.013320,9608,0.541792,-0.369464,936,0.718595,-0.621613,91,1.092128,-1.016268,0.631940,6,0,3266,0.693465,0.861978,-0.545993,0.774954,0.524495,1027,0.751693,0.983762,-0.656582,0.826680,0.504382,408,615,619,0.398827,0.397274,0.398049,1027,0.751693,0.983762,-0.656582,0.210321,0.826680,0.504382,518,0,0.000000,0,0.000000,2.013232,439,0.837292,1.071589,-0.715787,0.161731,0.833713,0.535308,235,0,0.000000,0,0.000000,2.167225,91,1.092128,1.371280,-1.016268,0.285714,0.758242,0.659341,60,0,0.000000,0,0.000000,2.446532,0.010256,0.020861,0.010588,770,3633,257,0.174881,0.749757,455,759,572,0.374794,0.443038,0.001311,0.001885,0.000938,35,251,56,0.122378,0.384615,28,101,63,0.217054,0.307692,0.344289,0.000000,0.000000,0.020861,0.683809,0.673056,0.654763,0.704929,0.730881,0.747511
1,entropy_prior,entropy_tail,0.100000,0.500000,4.000000,0.363508,0.487541,0.091341,0.383203,0.407153,0.089295,96993,5190,0.769729,0.769729,20718,0.440300,0.404099,60450,0.266804,0.015210,9608,0.543338,-0.387382,936,0.738868,-0.650785,91,1.102954,-1.054290,0.643666,6,0,3266,0.705020,0.871153,-0.571962,0.792100,0.545622,1027,0.771129,0.999779,-0.686538,0.839338,0.531646,381,535,646,0.415939,0.370983,0.392177,1027,0.771129,0.999779,-0.686538,0.202532,0.839338,0.531646,518,0,0.000000,28,0.019436,2.032054,439,0.853841,1.089945,-0.747068,0.157175,0.835991,0.558087,235,0,0.000000,10,0.016550,2.173298,91,1.102954,1.394741,-1.054290,0.285714,0.758242,0.659341,60,0,0.000000,0,0.010826,2.458693,0.009904,0.020068,0.010588,764,3495,263,0.179385,0.743914,425,642,602,0.398313,0.413827,0.001279,0.001813,0.000938,34,239,57,0.124542,0.373626,28,99,63,0.220472,0.307692,0.344289,0.000000,0.079727,0

,method,tau,overall_mae,overall_rmse,overall_bias,accuracy_0.2,underestimate_rate,underestimate_0.5_rate,overall_count,count_0_1,mae_0_1,bias_0_1,count_1_2,mae_1_2,bias_1_2,count_2_3,mae_2_3,bias_2_3,count_3_4,mae_3_4,bias_3_4,count_4_5,mae_4_5,bias_4_5,count_5_6,mae_5_6,bias_5_6,balanced_bin_mae,balanced_bin_groups_present,outside_broad_bins_count,tail35_count,tail35_mae,tail35_rmse,tail35_bias,tail35_underestimate_rate,tail35_underestimate_0.5_rate,tail40_count,tail40_mae,tail40_rmse,tail40_bias,tail40_underestimate_rate,tail40_underestimate_0.5_rate,m4_TP,m4_FP,m4_FN,m4_precision,m4_recall,m4_f1,extreme40_count,extreme40_mae,extreme40_rmse,extreme40_bias,extreme40_accuracy_0.2,extreme40_underestimate_rate,extreme40_underestimate_0.5_rate,extreme40_raw_severe_count,extreme40_recovered_count,extreme40_recovery_rate,extreme40_new_severe_count,extreme40_mean_abs_error_change,extreme40_p95_abs_error,extreme45_count,extreme45_mae,extreme45_rmse,extreme45_bias,extreme45_accuracy_0.2,extreme45_underestimate_rate,extreme45_underestimate_0.5_rate,extreme45_raw_severe_count,extreme45_recovered_count,extreme45_recovery_rate,extreme45_new_severe_count,extreme45_mean_abs_error_change,extreme45_p95_abs_error,extreme50_count,extreme50_mae,extreme50_rmse,extreme50_bias,extreme50_accuracy_0.2,extreme50_underestimate_rate,extreme50_underestimate_0.5_rate,extreme50_raw_severe_count,extreme50_recovered_count,extreme50_recovery_rate,extreme50_new_severe_count,extreme50_mean_abs_error_change,extreme50_p95_abs_error,m4_probability_brier,m4_mean_probability,m4_observed_frequency,m4_prob10_TP,m4_prob10_FP,m4_prob10_FN,m4_prob10_precision,m4_prob10_recall,m4_prob50_TP,m4_prob50_FP,m4_prob50_FN,m4_prob50_precision,m4_prob50_recall,m5_probability_brier,m5_mean_probability,m5_observed_frequency,m5_prob10_TP,m5_prob10_FP,m5_prob10_FN,m5_prob10_precision,m5_prob10_recall,m5_prob50_TP,m5_prob50_FP,m5_prob50_FN,m5_prob50_precision,m5_prob50_recall,argmax_overall_mae,argmax_M4_mae,argmax_M4_bias,prior_sigma,boost_cap_ratio,group_threshold
0,raw,0.000000,0.363262,0.488058,0.085835,0.384409,0.411566,0.091543,96993,5190,0.738733,0.738733,20718,0.425762,0.381808,60450,0.274630,0.013320,9608,0.541792,-0.369464,936,0.718595,-0.621613,91,1.092128,-1.016268,0.631940,6,0,3266,0.693465,0.861978,-0.545993,0.774954,0.524495,1027,0.751693,0.983762,-0.656582,0.826680,0.504382,408,615,619,0.398827,0.397274,0.398049,1027,0.751693,0.983762,-0.656582,0.210321,0.826680,0.504382,518,0,0.000000,0,0.000000,2.013232,439,0.837292,1.071589,-0.715787,0.161731,0.833713,0.535308,235,0,0.000000,0,0.000000,2.167225,91,1.092128,1.371280,-1.016268,0.285714,0.758242,0.659341,60,0,0.000000,0,0.000000,2.446532,0.010256,0.020861,0.010588,770,3633,257,0.174881,0.749757,455,759,572,0.374794,0.443038,0.001311,0.001885,0.000938,35,251,56,0.122378,0.384615,28,101,63,0.217054,0.307692,0.390698,0.858861,-0.489338,NaN,NaN,NaN
1,supported_tau_0,0.000000,0.362893,0.486541,0.085334,0.384471,0.411751,0.091543,96993,5190,0.738733,0.738733,20718,0.425558,0.381604,60450,0.274333,0.013023,9608,0.541541,-0.369720,936,0.702797,-0.637410,91,1.131215,-1.116597,0.635696,6,0,3266,0.689869,0.859761,-0.553474,0.780465,0.524495,1027,0.740758,0.978291,-0.679870,0.844206,0.504382,408,615,619,0.398827,0.397274,0.398049,1027,0.740758,0.978291,-0.679870,0.216164,0.844206,0.504382,518,0,0.000000,0,-0.010934,2.013232,439,0.816366,1.062552,-0.765612,0.175399,0.874715,0.535308,235,0,0.000000,0,-0.020925,2.167225,91,1.131215,1.378407,-1.116597,0.076923,0.956044,0.659341,60,0,0.000000,0,0.039087,2.446532,0.010205,0.020778,0.010588,770,3633,257,0.174881,0.749757,455,759,572,0.374794,0.443038,0.001190,0.001773,0.000938,35,251,56,0.122378,0.384615,28,98,63,0.222222,0.307692,0.389933,0.822347,-0.525852,NaN,NaN,NaN
2,logit_adjustment,0.050000,0.363922,0.488566,0.084501,0.383141,0.413123,0.092182,96993,5190,0.722371,0.722371,20718,0.417303,0.368670,60450,0.280817,0.014162,9608,0.539642,-0.352422,936,0.689347,-0.603962,91,1.071682,-1.0

In [19]:
ranking_metrics = ["overall_mae", "balanced_bin_mae", "tail40_mae", "tail40_bias"]

for name, results_df, parameters in (
    ("Entropy", entropy_results_df, ["gate", "alpha_max", "power"]),
    ("Logit adjustment", logit_adjustment_results_df, ["tau", "prior_sigma", "boost_cap_ratio", "group_threshold"])
):
    columns = [
        "method", *parameters, "overall_mae", "overall_rmse",
        "balanced_bin_mae", "tail40_count", "tail40_mae",
        "tail40_bias", "m4_precision", "m4_recall", "m4_FP"
    ]

    for metric in ranking_metrics:
        print(f"{name}: lowest {'absolute ' if metric == 'tail40_bias' else ''}{metric}")

        sorted_df = results_df.sort_values(
            metric,
            key=lambda x: x.abs() if metric == "tail40_bias" else x,
            na_position="last"
        )

        display(sorted_df[columns].head(3))

Entropy: lowest overall_mae


,method,gate,alpha_max,power,overall_mae,overall_rmse,balanced_bin_mae,tail40_count,tail40_mae,tail40_bias,m4_precision,m4_recall,m4_FP
14,gate_ablation,entropy_only,0.2,1.0,0.362489,0.483523,0.690961,1027,0.825679,-0.804073,0.478462,0.302824,339
3,entropy_prior,entropy_tail,0.1,2.0,0.363160,0.487404,0.638791,1027,0.764169,-0.676285,0.407563,0.377799,564
0,raw,entropy_tail,0.0,1.0,0.363262,0.488058,0.631940,1027,0.751693,-0.656582,0.398827,0.397274,615


Entropy: lowest balanced_bin_mae


,method,gate,alpha_max,power,overall_mae,overall_rmse,balanced_bin_mae,tail40_count,tail40_mae,tail40_bias,m4_precision,m4_recall,m4_FP
0,raw,entropy_tail,0.0,1.0,0.363262,0.488058,0.631940,1027,0.751693,-0.656582,0.398827,0.397274,615
3,entropy_prior,entropy_tail,0.1,2.0,0.363160,0.487404,0.638791,1027,0.764169,-0.676285,0.407563,0.377799,564
2,entropy_prior,entropy_tail,0.1,1.0,0.363342,0.487452,0.641714,1027,0.768432,-0.682598,0.409432,0.371957,551


Entropy: lowest tail40_mae


...

,method,tau,prior_sigma,boost_cap_ratio,group_threshold,overall_mae,overall_rmse,balanced_bin_mae,tail40_count,tail40_mae,tail40_bias,m4_precision,m4_recall,m4_FP
10,logit_adjustment,0.75,NaN,NaN,NaN,0.425871,0.583094,0.536546,1027,0.584282,-0.231133,0.219858,0.694255,2530
11,logit_adjustment,1.00,NaN,NaN,NaN,0.489180,0.665095,0.542175,1027,0.582593,-0.105004,0.183321,0.744888,3408
9,logit_adjustment,0.50,NaN,NaN,NaN,0.390417,0.531883,0.558435,1027,0.616277,-0.365243,0.262235,0.626095,1809


Logit adjustment: lowest tail40_mae


,method,tau,prior_sigma,boost_cap_ratio,group_threshold,overall_mae,overall_rmse,balanced_bin_mae,tail40_count,tail40_mae,tail40_bias,m4_precision,m4_recall,m4_FP
34,group_inverse,1.0,NaN,NaN,4.0,0.457509,0.630258,0.579208,1027,0.464450,-0.154318,0.150446,0.788705,4574
30,group_inverse,1.0,NaN,NaN,3.5,0.450162,0.603684,0.585660,1027,0.500634,-0.329511,0.244320,0.638754,2029
11,logit_adjustment,1.0,NaN,NaN,NaN,0.489180,0.665095,0.542175,1027,0.582593,-0.105004,0.183321,0.744888,3408


Logit adjustment: lowest absolute tail40_bias


,method,tau,prior_sigma,boost_cap_ratio,group_threshold,overall_mae,overall_rmse,balanced_bin_mae,tail40_count,tail40_mae,tail40_bias,m4_precision,m4_recall,m4_FP
11,logit_adjustment,1.00,NaN,NaN,NaN,0.489180,0.665095,0.542175,1027,0.582593,-0.105004,0.183321,0.744888,3408
34,group_inverse,1.00,NaN,NaN,4.0,0.457509,0.630258,0.579208,1027,0.464450,-0.154318,0.150446,0.788705,4574
10,logit_adjustment,0.75,NaN,NaN,NaN,0.425871,0.583094,0.536546,1027,0.584282,-0.231133,0.219858,0.694255,2530


## C. Fixed comparisons: where extremes improve or deteriorate

All configurations below are fixed in advance, not selected from the preceding rankings. Raw-entropy thirds and raw tail-probability groups are held fixed across methods, making changes comparable. True magnitudes define diagnostic cohorts only; they never gate inference. Counts are waveform counts, not independent earthquake counts. `delta_mae_vs_raw < 0` means improvement; recovery means a raw error ≤ −0.5 becomes > −0.5. This does not imply a correct prediction. The auxiliary `extreme_diagnostics_df` contains one row per configuration and non-overlapping cohort within each stratification; zero-count cohorts remain visible. No bootstrap or significance claim is made without event IDs.

In [20]:
import pandas as pd
import torch


def diagnostic_configurations():
    yield "raw", raw_probs
    yield "supported_tau_0", apply_logit_adjustment(val_logits, empirical_prior, 0.0)
    yield "entropy_a0.4_full", apply_entropy_prior(val_logits, smoothed_prior, BIN_CENTERS, 0.4, 1.0, 4.0)[0]
    yield "entropy_a0.4_entropy_only", apply_entropy_prior(val_logits, smoothed_prior, BIN_CENTERS, 0.4, 1.0, 4.0, "entropy_only")[0]
    yield "inverse_tau0.25", apply_logit_adjustment(val_logits, empirical_prior, 0.25)
    yield "smoothed_sigma1_tau0.25", apply_logit_adjustment(val_logits, smoothed_supported_prior(1.0), 0.25)
    yield "capped_ratio2_tau0.25", capped_logit_adjustment(val_logits, empirical_prior, 0.25, 2.0)
    yield "group_M4_tau0.25", group_logit_adjustment(val_logits, empirical_prior, 0.25, 4.0)


entropy_cutpoints = torch.quantile(normalized_entropy, torch.tensor([1 / 3, 2 / 3], dtype=normalized_entropy.dtype))

entropy_groups = (
    ("low", normalized_entropy <= entropy_cutpoints[0]),
    ("middle", (normalized_entropy > entropy_cutpoints[0]) & (normalized_entropy <= entropy_cutpoints[1])),
    ("high", normalized_entropy > entropy_cutpoints[1])
)

tail_groups = (
    ("below_0.1", tail_probability < 0.1),
    ("0.1_to_0.5", (tail_probability >= 0.1) & (tail_probability < 0.5)),
    ("at_least_0.5", tail_probability >= 0.5)
)

magnitude_groups = (
    ("below_3.5", val_targets < 3.5),
    ("3.5_to_4", (val_targets >= 3.5) & (val_targets < 4)),
    ("4_to_5", (val_targets >= 4) & (val_targets < 5)),
    ("at_least_5", val_targets >= 5)
)

raw_errors = raw_predictions - val_targets
diagnostic_rows = []

for configuration, probs in diagnostic_configurations():
    predictions = probs @ BIN_CENTERS
    errors = predictions - val_targets

    for stratification, groups in (("raw_entropy", entropy_groups), ("raw_M4_probability", tail_groups)):
        for magnitude_group, magnitude_mask in magnitude_groups:
            for raw_group, raw_mask in groups:
                mask = magnitude_mask & raw_mask
                count = int(mask.sum())
                severe = mask & (raw_errors <= -0.5)
                recovered = int((severe & (errors > -0.5)).sum())
                metrics = calculate_metrics(predictions[mask], val_targets[mask])

                diagnostic_rows.append({
                    "configuration": configuration,
                    "stratification": stratification,
                    "true_magnitude_group": magnitude_group,
                    "raw_group": raw_group,
                    "count": count,
                    **metrics,
                    "raw_mae": raw_errors[mask].abs().mean().item() if count else None,
                    "delta_mae_vs_raw": (errors[mask].abs() - raw_errors[mask].abs()).mean().item() if count else None,
                    "mean_prediction_shift": (predictions[mask] - raw_predictions[mask]).mean().item() if count else None,
                    "improved_count": int((errors[mask].abs() < raw_errors[mask].abs()).sum()),
                    "worsened_count": int((errors[mask].abs() > raw_errors[mask].abs()).sum()),
                    "raw_severe_count": int(severe.sum()),
                    "recovered_count": recovered,
                    "recovery_rate": recovered / int(severe.sum()) if severe.any() else None,
                    "new_severe_count": int((mask & (raw_errors > -0.5) & (errors <= -0.5)).sum())
                })


extreme_diagnostics_df = pd.DataFrame(diagnostic_rows)

display(
    extreme_diagnostics_df.loc[
        extreme_diagnostics_df["true_magnitude_group"].isin(["4_to_5", "at_least_5"])
        & extreme_diagnostics_df["stratification"].eq("raw_M4_probability")
        & extreme_diagnostics_df["count"].gt(0)
    ]
)

,configuration,stratification,true_magnitude_group,raw_group,count,mae,rmse,bias,accuracy_0.2,underestimate_rate,underestimate_0.5_rate,raw_mae,delta_mae_vs_raw,mean_prediction_shift,improved_count,worsened_count,raw_severe_count,recovered_count,recovery_rate,new_severe_count
18,raw,raw_M4_probability,4_to_5,below_0.1,234,1.570952,1.628789,-1.570952,0.000000,1.000000,1.000000,1.570952,0.000000e+00,0.000000e+00,0,0,234,0,0.000000,0
19,raw,raw_M4_probability,4_to_5,0.1_to_0.5,296,0.628074,0.710418,-0.628074,0.118243,1.000000,0.631757,0.628074,0.000000e+00,0.000000e+00,0,0,187,0,0.000000,0
20,raw,raw_M4_probability,4_to_5,at_least_0.5,406,0.293330,0.360008,-0.069746,0.381773,0.615764,0.091133,0.293330,0.000000e+00,0.000000e+00,0,0,37,0,0.000000,0
21,raw,raw_M4_probability,at_least_5,below_0.1,23,2.244638,2.255627,-2.244638,0.000000,1.000000,1.000000,2.244638,0.000000e+00,0.000000e+00,0,0,23,0,0.000000,0
22,raw,raw_M4_probability,at_least_5,0.1_to_0.5,19,1.412011,1.429666,-1.412011,0.000000,1.000000,1.000000,1.412011,0.000000e+00,0.000000e+00,0,0,19,0,0.000000,0
23,raw,raw_M4_probability,at_least_5,at_least_0.5,49,0.427118,0.558090,-0.286234,0.530612,0.551020,0.367347,0.427118,0.000000e+00,0.000000e+00,0,0,18,0,0.000000,0
42,supported_tau_0,raw_M4_probability,4_to_5,below_0.1,234,1.570952,1.628789,-1.570952,0.000000,1.000000,1.000000,1.570952,4.394032e-08,-4.394032e-08,1,165,234,0,0.000000,0
43,supported_tau_0,raw_M4_probability,4_to_5,0.1_to_0.5,296,0.628074,0.710418,-0.628074,0.118243,1.000000,0.631757,0.628074,3.968459e-08,-3.968459e-08,1,196,187,0,0.000000,0
44,supported_tau_0,raw_M4_probability,4_to_5,at_least_0.5,406,0.256911,0.313144,-0.106166,0.443350,0.615764,0.091133,0.293330,-3.641937e-02,-3.641979e-02,109,119,37,0,0.000000,0
45,supported_tau_0,raw_M4_probability,at_least_5,below_0.1,23,2.244638,2.255627,-2.244638,0.000000,1.000000,1.000000,2.244638,4.805798e-12,-4.805798e-12,0,16,23,0,0.000000,0


In [ ]:
entropy_results_df.to_csv( "entropy_results5s.csv", index=False)
logit_adjustment_results_df.to_csv( "logit_adjustment_results5s.csv", index=False)
extreme_diagnostics_df.to_csv("extreme_diagnostics5s.csv", index=False)

In [ ]:
import numpy as np

best_predictions = best_probs @ BIN_CENTERS

np.savez(
    "best_method_predictions.npz",
    targets=val_targets.numpy(),
    raw_predictions=raw_predictions.numpy(),
    corrected_predictions=best_predictions.numpy(),
    raw_errors=(raw_predictions - val_targets).numpy(),
    corrected_errors=(best_predictions - val_targets).numpy(),
    entropy=normalized_entropy.numpy(),
    tail_probability=tail_probability.numpy()
)